# Week 12 Lab — Big Data Tools: SQL · APIs · PySpark
## Student: Didier Ndahimana
## Student ID: 29254
## Program: Day
---


## Setup and Imports

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import sqlite3
import json
import requests
from datetime import datetime

# Install PySpark if needed (uncomment for Colab)
# !pip install pyspark

---
# Part 1 · SQL with Python (12 marks)
A ready-made cell loads the CSV into an SQLite database (momo.db, table transactions). Answer with pd.read_sql(query, conn).

### Load CSV into SQLite Database

In [2]:
# Load the CSV file
df = pd.read_csv('week12_transactions.csv')

# Create SQLite database and load data
conn = sqlite3.connect('momo.db')
df.to_sql('transactions', conn, if_exists='replace', index=False)

print("Database created successfully!")
print(f"Total records loaded: {len(df)}")
print("\nFirst 5 rows:")
print(df.head())

Database created successfully!
Total records loaded: 2000

First 5 rows:
   txn_id        date  hour    district txn_type  amount_rwf  flagged
0  T00001  2026-11-01     8      Gasabo     send        1000        0
1  T00002  2026-11-09    12  Nyarugenge     send       25000        0
2  T00003  2026-11-19     8     Muhanga     send       51000        0
3  T00004  2026-11-01    17        Huye     send       79000        0
4  T00005  2026-11-15    12        Huye     send      109000        0


### Exercise 1.1 — SELECT / WHERE / ORDER BY (4 marks)
(a) In SQL: txn_id, district, amount_rwf of the 5 largest **send** transactions above 100,000 RWF. (b) Count the flagged transactions (flagged = 1).

**Checkpoint: 11 flagged.**

In [3]:
# Part (a): 5 largest send transactions above 100,000 RWF
query_1a = """
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE txn_type = 'send' AND amount_rwf > 100000
ORDER BY amount_rwf DESC
LIMIT 5
"""

result_1a = pd.read_sql(query_1a, conn)
print("Exercise 1.1(a): 5 largest send transactions above 100,000 RWF")
print("="*60)
print(result_1a)
print()

Exercise 1.1(a): 5 largest send transactions above 100,000 RWF
   txn_id    district  amount_rwf
0  T01708    Kicukiro      140000
1  T00364        Huye      139000
2  T00895     Muhanga      139000
3  T00899  Nyarugenge      139000
4  T01314      Gasabo      139000



In [4]:
# Part (b): Count flagged transactions
query_1b = """
SELECT COUNT(*) as flagged_count
FROM transactions
WHERE flagged = 1
"""

result_1b = pd.read_sql(query_1b, conn)
print("Exercise 1.1(b): Count of flagged transactions")
print("="*60)
print(f"Flagged transactions: {result_1b['flagged_count'].iloc[0]}")
print("Checkpoint: 11 flagged ✓")

Exercise 1.1(b): Count of flagged transactions
Flagged transactions: 11
Checkpoint: 11 flagged ✓


### Exercise 1.2 — GROUP BY (5 marks)
(a) Per txn_type: COUNT(*), SUM and AVG of amount_rwf, ordered by total descending — which type moves the most money? (b) Total per district — which district leads?

**Checkpoint: 'send' moves ~62.0M RWF; Muhanga leads with ~21.1M.**

In [5]:
# Part (a): Per txn_type statistics
query_2a = """
SELECT 
    txn_type,
    COUNT(*) as transaction_count,
    SUM(amount_rwf) as total_amount,
    AVG(amount_rwf) as average_amount
FROM transactions
GROUP BY txn_type
ORDER BY total_amount DESC
"""

result_2a = pd.read_sql(query_2a, conn)
print("Exercise 1.2(a): Statistics per transaction type")
print("="*80)
print(result_2a)
print(f"\n→ '{result_2a.iloc[0]['txn_type']}' moves the most money: {result_2a.iloc[0]['total_amount']:,.2f} RWF")
print("Checkpoint: 'send' moves ~62.0M RWF ✓")
print()

Exercise 1.2(a): Statistics per transaction type
  txn_type  transaction_count  total_amount  average_amount
0     send                868      62049000    71485.023041
1  paybill                330      29970000    90818.181818
2  receive                489      27024000    55263.803681
3  airtime                313        833000     2661.341853

→ 'send' moves the most money: 62,049,000.00 RWF
Checkpoint: 'send' moves ~62.0M RWF ✓



In [6]:
# Part (b): Total per district
query_2b = """
SELECT 
    district,
    SUM(amount_rwf) as total_amount
FROM transactions
GROUP BY district
ORDER BY total_amount DESC
"""

result_2b = pd.read_sql(query_2b, conn)
print("Exercise 1.2(b): Total per district")
print("="*60)
print(result_2b)
print(f"\n→ '{result_2b.iloc[0]['district']}' leads with {result_2b.iloc[0]['total_amount']:,.2f} RWF")
print("Checkpoint: Muhanga leads with ~21.1M ✓")

Exercise 1.2(b): Total per district
     district  total_amount
0     Muhanga      21076000
1      Gasabo      20738000
2        Huye      18736000
3  Nyarugenge      18091000
4    Kicukiro      11338000
5     Musanze      10490500
6      Rubavu      10110500
7   Nyagatare       9296000

→ 'Muhanga' leads with 21,076,000.00 RWF
Checkpoint: Muhanga leads with ~21.1M ✓


### Exercise 1.3 — The busiest hour (3 marks)
(a) In SQL: transactions per hour, busiest first. (b) Answer the same question in Pandas and confirm both agree.

**Checkpoint: hour 20, with 185 transactions.**

In [7]:
# Part (a): SQL - Transactions per hour
query_3a = """
SELECT 
    hour,
    COUNT(*) as transaction_count
FROM transactions
GROUP BY hour
ORDER BY transaction_count DESC
"""

result_3a = pd.read_sql(query_3a, conn)
print("Exercise 1.3(a): Transactions per hour (SQL)")
print("="*60)
print(result_3a.head(10))
print(f"\n→ Busiest hour: {result_3a.iloc[0]['hour']} with {result_3a.iloc[0]['transaction_count']} transactions")
print("Checkpoint: hour 20, with 185 transactions ✓")
print()

Exercise 1.3(a): Transactions per hour (SQL)
   hour  transaction_count
0    20                185
1    17                183
2    18                176
3    12                170
4    11                169
5    10                168
6     8                168
7    19                158
8     9                144
9    16                 38

→ Busiest hour: 20 with 185 transactions
Checkpoint: hour 20, with 185 transactions ✓



In [8]:
# Part (b): Pandas - Transactions per hour
result_3b = df.groupby('hour').size().reset_index(name='transaction_count')
result_3b = result_3b.sort_values('transaction_count', ascending=False)

print("Exercise 1.3(b): Transactions per hour (Pandas)")
print("="*60)
print(result_3b.head(10))
print(f"\n→ Busiest hour: {result_3b.iloc[0]['hour']} with {result_3b.iloc[0]['transaction_count']} transactions")

# Confirm both agree
sql_busiest = result_3a.iloc[0]['hour']
pandas_busiest = result_3b.iloc[0]['hour']
print(f"\n✓ SQL and Pandas agree: Both show hour {sql_busiest} as the busiest")

Exercise 1.3(b): Transactions per hour (Pandas)
    hour  transaction_count
20    20                185
17    17                183
18    18                176
12    12                170
11    11                169
10    10                168
8      8                168
19    19                158
9      9                144
16    16                 38

→ Busiest hour: 20 with 185 transactions

✓ SQL and Pandas agree: Both show hour 20 as the busiest


---
# Part 2 · APIs & JSON (10 marks)

### Exercise 2.1 — JSON by hand (3 marks)
Given the JSON string in the notebook, parse it with json.loads and print: (a) the city, (b) the number of readings, (c) their average (2 decimals).

In [9]:
# Sample JSON string (provided in the lab)
json_string = '''{
  "city": "Kigali",
  "readings": [22.5, 23.1, 24.3, 22.8, 23.5]
}'''

# Parse JSON
data = json.loads(json_string)

# Extract information
city = data['city']
readings = data['readings']
num_readings = len(readings)
average = sum(readings) / num_readings

print("Exercise 2.1: JSON parsing")
print("="*60)
print(f"(a) City: {city}")
print(f"(b) Number of readings: {num_readings}")
print(f"(c) Average: {average:.2f}")

Exercise 2.1: JSON parsing
(a) City: Kigali
(b) Number of readings: 5
(c) Average: 23.24


### Exercise 2.2 — Call a real API (4 marks)
A provided cell fetches Kigali's 24-hour forecast from Open-Meteo (falling back to week12_api_sample.json offline). From data, extract the temperature and time lists and print: (a) how many hours, (b) max and min temperature, (c) the TIME of the hottest hour (.index(max(...)) — Week 5!).

In [10]:
# Fetch API data (with fallback to offline file)
try:
    # Try to fetch from API
    url = ("https://api.open-meteo.com/v1/forecast"
           "?latitude=-1.9441&longitude=30.0619"
           "&hourly=temperature_2m&forecast_days=1&timezone=Africa%2FKigali")
    response = requests.get(url, timeout=10)
    data = response.json()
    print("✓ API data fetched successfully from Open-Meteo")
except Exception as e:
    # Fallback to offline sample
    print("⚠ API unavailable, using offline sample data (week12_api_sample.json)")
    with open('week12_api_sample.json', 'r') as f:
        data = json.load(f)

# Extract time and temperature lists
time_list = data['hourly']['time']
temp_list = data['hourly']['temperature_2m']

# Calculate statistics
num_hours = len(time_list)
max_temp = max(temp_list)
min_temp = min(temp_list)
hottest_index = temp_list.index(max_temp)
hottest_time = time_list[hottest_index]

print("\nExercise 2.2: API data analysis")
print("="*60)
print(f"(a) Number of hours: {num_hours}")
print(f"(b) Max temperature: {max_temp}°C")
print(f"    Min temperature: {min_temp}°C")
print(f"(c) Time of hottest hour: {hottest_time}")

✓ API data fetched successfully from Open-Meteo

Exercise 2.2: API data analysis
(a) Number of hours: 24
(b) Max temperature: 26.2°C
    Min temperature: 14.9°C
(c) Time of hottest hour: 2026-11-25T14:00


### Exercise 2.3 — API data meets your toolbox (3 marks)
Build a DataFrame with columns time and temp_c from the API data. (a) describe() the temperature. (b) Count hours ≥ 25 °C with a mask.

In [11]:
# Build DataFrame from API data
api_df = pd.DataFrame({
    'time': time_list,
    'temp_c': temp_list
})

print("Exercise 2.3: API data in DataFrame")
print("="*60)

# Part (a): Describe the temperature
print("(a) Temperature statistics:")
print(api_df['temp_c'].describe())

# Part (b): Count hours >= 25°C
mask = api_df['temp_c'] >= 25
count_hot_hours = mask.sum()

print(f"\n(b) Hours with temperature ≥ 25°C: {count_hot_hours}")

# Display the DataFrame
print("\nDataFrame preview:")
print(api_df.head(10))

Exercise 2.3: API data in DataFrame
(a) Temperature statistics:
count    24.000000
mean     19.883333
std       3.789880
min      14.900000
25%      16.375000
50%      19.350000
75%      22.850000
max      26.200000
Name: temp_c, dtype: float64

(b) Hours with temperature ≥ 25°C: 3

DataFrame preview:
               time  temp_c
0  2026-11-25T00:00    16.5
1  2026-11-25T01:00    16.0
2  2026-11-25T02:00    16.0
3  2026-11-25T03:00    15.4
4  2026-11-25T04:00    15.1
5  2026-11-25T05:00    14.9
6  2026-11-25T06:00    14.9
7  2026-11-25T07:00    17.4
8  2026-11-25T08:00    20.7
9  2026-11-25T09:00    22.4


---
# Part 3 · PySpark (10 marks)
A provided cell starts a local SparkSession. Startup warnings are normal.

### Initialize PySpark

In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, desc

# Create Spark session
spark = SparkSession.builder \
    .appName("Week12_Lab") \
    .master("local[*]") \
    .getOrCreate()

print("✓ Spark session started successfully")
print(f"Spark version: {spark.version}")

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


✓ Spark session started successfully
Spark version: 4.2.0


### Exercise 3.1 — Load & inspect (3 marks)
Load the CSV into a Spark DataFrame sdf with spark.read.csv(..., header=True, inferSchema=True). Print the row count, the schema, and the first 5 rows.

In [13]:
# Load CSV into Spark DataFrame
sdf = spark.read.csv('week12_transactions.csv', header=True, inferSchema=True)

print("Exercise 3.1: Load & inspect Spark DataFrame")
print("="*60)

# Print row count
row_count = sdf.count()
print(f"Row count: {row_count}")

# Print schema
print("\nSchema:")
sdf.printSchema()

# Show first 5 rows
print("\nFirst 5 rows:")
sdf.show(5)

Exercise 3.1: Load & inspect Spark DataFrame
Row count: 2000

Schema:
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)


First 5 rows:
+------+----------+----+----------+--------+----------+-------+
|txn_id|      date|hour|  district|txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|   8|    Gasabo|    send|      1000|      0|
|T00002|2026-11-09|  12|Nyarugenge|    send|     25000|      0|
|T00003|2026-11-19|   8|   Muhanga|    send|     51000|      0|
|T00004|2026-11-01|  17|      Huye|    send|     79000|      0|
|T00005|2026-11-15|  12|      Huye|    send|    109000|      0|
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.2 — Filter (3 marks)
(a) Count transactions above 100,000 RWF with sdf.filter(...). (b) Show 5 flagged transactions.

**Checkpoint: 451 transactions above 100K.**

In [14]:
# Part (a): Count transactions above 100,000 RWF
count_above_100k = sdf.filter(col('amount_rwf') > 100000).count()

print("Exercise 3.2: Filter operations")
print("="*60)
print(f"(a) Transactions above 100,000 RWF: {count_above_100k}")
print("Checkpoint: 451 transactions above 100K ✓")
print()

Exercise 3.2: Filter operations
(a) Transactions above 100,000 RWF: 451
Checkpoint: 451 transactions above 100K ✓



In [15]:
# Part (b): Show 5 flagged transactions
print("(b) 5 flagged transactions:")
sdf.filter(col('flagged') == 1).show(5)

(b) 5 flagged transactions:
+------+----------+----+--------+--------+----------+-------+
|txn_id|      date|hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|  19| Muhanga|    send|    127000|      1|
|T00326|2026-11-07|  22|    Huye|    send|    127000|      1|
|T00536|2026-11-27|  17|  Gasabo|    send|    127000|      1|
|T00655|2026-11-30|  17|  Rubavu|    send|    134000|      1|
|T00697|2026-11-01|   9| Muhanga|    send|    127000|      1|
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


### Exercise 3.3 — groupBy, fourth reunion (4 marks)
(a) Total amount per txn_type with groupBy(...).sum(...) and .show(). (b) Per-district totals converted with .toPandas() and sorted. (c) In a comment: do the Spark answers match your SQL answers? (They must.)

In [16]:
# Part (a): Total amount per txn_type
print("Exercise 3.3: groupBy operations")
print("="*60)
print("(a) Total amount per txn_type:")
sdf.groupBy('txn_type').sum('amount_rwf').show()

Exercise 3.3: groupBy operations
(a) Total amount per txn_type:
+--------+---------------+
|txn_type|sum(amount_rwf)|
+--------+---------------+
| paybill|       29970000|
| airtime|         833000|
| receive|       27024000|
|    send|       62049000|
+--------+---------------+



In [17]:
# Part (b): Per-district totals converted to Pandas and sorted
print("(b) Per-district totals (sorted):")
district_totals = sdf.groupBy('district').sum('amount_rwf').toPandas()
district_totals.columns = ['district', 'total_amount']
district_totals = district_totals.sort_values('total_amount', ascending=False)
print(district_totals)

print("\n" + "="*60)
print("(c) Verification: Do Spark answers match SQL answers?")
print("="*60)
print("✓ YES! The Spark results match the SQL results from Part 1:")
print("  - 'send' transactions total ~62.0M RWF (both methods)")
print("  - Muhanga district leads with ~21.1M RWF (both methods)")
print("  - Transaction counts and totals are identical")
print("\nThis demonstrates that Spark and SQL produce the same results,")
print("but Spark can scale to much larger datasets across distributed clusters!")

(b) Per-district totals (sorted):


C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


     district  total_amount
4     Muhanga      21076000
5      Gasabo      20738000
6        Huye      18736000
1  Nyarugenge      18091000
0    Kicukiro      11338000
2     Musanze      10490500
7      Rubavu      10110500
3   Nyagatare       9296000

(c) Verification: Do Spark answers match SQL answers?
✓ YES! The Spark results match the SQL results from Part 1:
  - 'send' transactions total ~62.0M RWF (both methods)
  - Muhanga district leads with ~21.1M RWF (both methods)
  - Transaction counts and totals are identical

This demonstrates that Spark and SQL produce the same results,
but Spark can scale to much larger datasets across distributed clusters!


---
# Bonus (+3) — Spark speaks SQL too
Register the view with sdf.createOrReplaceTempView("transactions") and run your Part 1 GROUP BY unchanged via spark.sql(...). In a comment: why is it powerful that the same SQL runs on SQLite and on a thousand-machine cluster?

In [18]:
# Register Spark DataFrame as SQL view
sdf.createOrReplaceTempView("transactions")

print("Bonus Exercise: Spark speaks SQL too")
print("="*60)

# Run the same GROUP BY query from Exercise 1.2(a)
spark_sql_query = """
SELECT 
    txn_type,
    COUNT(*) as transaction_count,
    SUM(amount_rwf) as total_amount,
    AVG(amount_rwf) as average_amount
FROM transactions
GROUP BY txn_type
ORDER BY total_amount DESC
"""

result_spark_sql = spark.sql(spark_sql_query)
print("\nRunning Part 1 GROUP BY query on Spark SQL:")
result_spark_sql.show()

print("\n" + "="*60)
print("WHY IS THIS POWERFUL?")
print("="*60)
print("""
It's powerful that the SAME SQL runs on SQLite and on a thousand-machine cluster because:

1. **Portability**: You can write SQL once and run it anywhere - from your laptop
   (SQLite) to massive distributed clusters (Spark). No need to rewrite code!

2. **Easy Scaling**: Start development with small datasets on SQLite, then move
   to Spark when data grows to terabytes WITHOUT changing your queries.

3. **Skill Transfer**: Analysts who know SQL can immediately work with Big Data
   on Spark clusters without learning new syntax or APIs.

4. **Automatic Optimization**: Spark's Catalyst optimizer distributes the work
   across thousands of machines, parallelizing operations automatically while
   you use familiar SQL.

5. **Cost Efficiency**: Test on small data locally (free), then scale to cloud
   clusters only when needed, using the exact same code.

This abstraction is the foundation of modern Big Data analytics - write once,
run anywhere, scale infinitely!
""")

Bonus Exercise: Spark speaks SQL too

Running Part 1 GROUP BY query on Spark SQL:
+--------+-----------------+------------+------------------+
|txn_type|transaction_count|total_amount|    average_amount|
+--------+-----------------+------------+------------------+
|    send|              868|    62049000| 71485.02304147465|
| paybill|              330|    29970000| 90818.18181818182|
| receive|              489|    27024000|  55263.8036809816|
| airtime|              313|      833000|2661.3418530351437|
+--------+-----------------+------------+------------------+


WHY IS THIS POWERFUL?

It's powerful that the SAME SQL runs on SQLite and on a thousand-machine cluster because:

1. **Portability**: You can write SQL once and run it anywhere - from your laptop
   (SQLite) to massive distributed clusters (Spark). No need to rewrite code!

2. **Easy Scaling**: Start development with small datasets on SQLite, then move
   to Spark when data grows to terabytes WITHOUT changing your queries.



---
# Part 4 · Choose Your Weapon (3 marks — written)
For each scenario, name the FIRST tool you'd reach for (Pandas / SQL / Spark / API) and justify in one sentence:

### Scenario 1: Ministry of Health gives you read-access to their PostgreSQL database and asks for monthly totals per district.

**Answer: SQL**

**Justification:** SQL is the native language of PostgreSQL databases, allowing direct, efficient querying at the source without loading data into memory, and the aggregation (GROUP BY) can be performed by the database server which is optimized for such operations.

### Scenario 2: Your project group has a 40,000-row CSV needing cleaning, three charts, and a regression model.

**Answer: Pandas**

**Justification:** Pandas excels at exploratory data analysis on medium-sized datasets (< 1GB), providing built-in functions for data cleaning, easy integration with matplotlib/seaborn for visualization, and seamless connection to scikit-learn for regression modeling.

### Scenario 3: A telecom wants nationwide call-record analysis — 40 TB already on their cluster.

**Answer: Spark**

**Justification:** Spark is specifically designed for distributed processing of massive datasets (terabytes to petabytes) across clusters, making it the only practical tool for 40TB of data that cannot fit in a single machine's memory.

### Scenario 4: Your dashboard must always show today's weather without anyone uploading a file.

**Answer: API**

**Justification:** APIs provide real-time, automatically updated data from external sources (like weather services) without manual file uploads, ensuring the dashboard always displays current information by fetching data programmatically on demand.

---
# Summary and Cleanup

In [19]:
# Close database connection
conn.close()

# Stop Spark session
spark.stop()

print("="*60)
print("Lab completed successfully!")
print("="*60)
print("\nSubmission Checklist:")
print("[ ] Notebook renamed Week12_YourName")
print("[ ] SQL and Spark answers agree")
print("[ ] API exercise ran (live or offline)")
print("[ ] Part 4 scenarios answered")
print("[ ] Project proposal written")
print("[ ] Screenshots taken")
print("[ ] Submitted before end of Week 12")
print("\n✓ All exercises completed!")

Lab completed successfully!

Submission Checklist:
[ ] Notebook renamed Week12_YourName
[ ] SQL and Spark answers agree
[ ] API exercise ran (live or offline)
[ ] Part 4 scenarios answered
[ ] Project proposal written
[ ] Screenshots taken
[ ] Submitted before end of Week 12

✓ All exercises completed!
